## Simulation with PCA — Real-Time IDS Pipeline

### Flow:
1. Load preprocessed data (`modified_daata.csv`)
2. Remove irrelevant columns (Flow ID, Source IP, Destination IP, Timestamp, Label)
3. Sample `n` rows to simulate incoming traffic
4. Load PCA + Scaler artifacts and transform data
5. Load XGBoost (PCA) model and predict
6. Decode encoded labels back to human-readable strings
7. Stream predictions to Power BI Streaming Dataset

In [ ]:
import pandas as pd
import numpy as np
import joblib
import requests
import json
import time
from xgboost import XGBClassifier

In [ ]:
DATA_PATH = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/Data/modified_daata.csv'
MODEL_DIR = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/Packages and Models'

df = pd.read_csv(DATA_PATH)
# Strip all column name whitespace upfront
df.columns = df.columns.str.strip()
print(f"Loaded data: {df.shape}")
df.head()

In [ ]:
cols_to_drop = ['Flow ID', 'Timestamp', 'Source IP', 'Destination IP', 'Label']
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"After dropping irrelevant cols: {df.shape}")
print("Remaining columns:", df.columns.tolist())

### Step: Sample Traffic — Simulating Incoming Network Flows
Set `n` to the number of flows you want to simulate. These rows are randomly sampled from the dataset to mimic live traffic arrival.

In [ ]:
n = 100  # Number of flows to simulate — change as needed
simulated_data = df.sample(n, random_state=None).reset_index(drop=True)
print(f"Simulated {n} traffic flows.")
simulated_data.head()

### Step: Load PCA Artifact and Transform
The PCA model was fit on training data only. We apply the same transform to keep consistency.

In [ ]:
pca = joblib.load(f'{MODEL_DIR}/pca.pkl')

# Validate features
try:
    expected_features = [c.strip() for c in pca.feature_names_in_]
    missing = set(expected_features) - set(simulated_data.columns)
    extra = set(simulated_data.columns) - set(expected_features)
    if missing:
        print(f"⚠️ Missing features: {missing}")
    if extra:
        print(f"ℹ️ Extra features (will be ignored by PCA): {extra}")
    # Reorder columns to match PCA training order
    simulated_data = simulated_data[expected_features]
except AttributeError:
    print("ℹ️ PCA artifact has no feature_names_in_ (older sklearn). Proceeding with current column order.")

df_pca_data = pca.transform(simulated_data)
df_pca = pd.DataFrame(df_pca_data, columns=[f'PCA_{i+1}' for i in range(df_pca_data.shape[1])])
print(f"PCA reduced dimensions: {simulated_data.shape[1]} → {df_pca.shape[1]}")
df_pca.head()

### Step: Scale PCA Components
A StandardScaler was fit on PCA-transformed training data. Apply the same scaler here.

In [ ]:
scaler = joblib.load(f'{MODEL_DIR}/scaler_pca.pkl')
scaled_data = scaler.transform(df_pca)
df_scaled = pd.DataFrame(scaled_data, columns=df_pca.columns)
print(f"Scaled shape: {df_scaled.shape}")
df_scaled.head()

### Step: Load XGBoost (PCA) Model and Predict

In [ ]:
xgb = XGBClassifier()
xgb.load_model(f'{MODEL_DIR}/xgb_pca.model')

predict_encoded = xgb.predict(df_scaled.values)
print(f"Predictions (encoded): {predict_encoded[:10]}")

### Step: Decode Predictions to Human-Readable Labels

In [ ]:
lb = joblib.load(f'{MODEL_DIR}/labelencoded.pkl')
predict_labels = lb.inverse_transform(predict_encoded)
print("Sample predictions:", predict_labels[:10])

# Add predictions to a results DataFrame
results_df = simulated_data.copy()
results_df['Predict_Label'] = predict_labels
print(f"\nAttack distribution in simulation:")
print(pd.Series(predict_labels).value_counts())

### Step: Stream Predictions to Power BI
Each row is sent as a JSON payload to the Power BI Streaming Dataset REST API.

⚠️ **Replace `powerbi_url`** with your own Power BI Streaming Dataset push URL before running.

In [ ]:
# ─── CONFIGURE YOUR POWER BI URL BELOW ───────────────────────────────────────
POWERBI_URL = "https://api.powerbi.com/beta/YOUR_TENANT_ID/datasets/YOUR_DATASET_ID/rows?experience=power-bi&key=YOUR_API_KEY"
# ─────────────────────────────────────────────────────────────────────────────

headers = {"Content-Type": "application/json"}

# Build streaming payload: PCA components + prediction label
pca_col_names = df_scaled.columns.tolist()

success_count = 0
fail_count = 0

for idx in range(len(df_scaled)):
    row = df_scaled.iloc[idx]
    
    # Build payload — convert numpy types to native Python for JSON serialization
    payload = {col: float(row[col]) for col in pca_col_names}
    payload["Predict_Label"] = str(predict_labels[idx])
    payload["Row_Index"] = int(idx)
    
    try:
        response = requests.post(POWERBI_URL, headers=headers, data=json.dumps([payload]), timeout=10)
        if response.status_code in [200, 202]:
            success_count += 1
            print(f"✅ Row {idx:3d} → {predict_labels[idx]}")
        else:
            fail_count += 1
            print(f"❌ Row {idx:3d} failed: {response.status_code} — {response.text[:80]}")
    except requests.exceptions.RequestException as e:
        fail_count += 1
        print(f"⚠️ Row {idx:3d} exception: {e}")
    
    time.sleep(0.05)  # Small delay to avoid rate-limiting

print(f"\n📊 Streaming complete: {success_count} sent, {fail_count} failed out of {len(df_scaled)} rows.")

### Summary
- Loaded `modified_daata.csv` and sampled `n` traffic flows
- Applied saved PCA transform (dimensionality reduction)
- Applied saved StandardScaler
- Predicted using XGBoost (PCA) model
- Decoded predictions using LabelEncoder
- Streamed each flow + label to Power BI Streaming Dataset

**Interpretation of predictions:**
- `BENIGN` → Normal network traffic
- Any other label → Detected intrusion/attack type